# superGroups · trancefusion guitarist

Original synthetic sixteen-bar exercises derived from the supplied musical description. No artist audio was used. Select a T4 GPU runtime. Three sequential training experiments compare event prediction, one-bar motif memory, and robustness to missing self-history. See the musician profile for representation limits.


In [ ]:
from pathlib import Path
import subprocess,sys,os,torch
assert torch.cuda.is_available(), 'Choose a GPU runtime'
ROOT=Path('/content/trancefusion-guitarist')
REPO=Path('/content/trancefusion-source')
if not REPO.exists(): subprocess.run(['git','clone','https://github.com/nicjams/nicjams.github.io.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','a3b2d8b'],check=True)
os.chdir(REPO/'superGroups')
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
def run(*args): subprocess.run([sys.executable,*map(str,args)],check=True)
print(torch.cuda.get_device_name(0))


In [ ]:
run('-m','supergroups.trancefusion','--out',ROOT/'run-a-event-baseline','--updates','1500','--device','cuda')
run('-m','supergroups.trancefusion_probe',ROOT/'run-a-event-baseline/best.pt','--out',ROOT/'run-a-event-baseline/validation-probe')


In [ ]:
run('-m','supergroups.trancefusion','--out',ROOT/'run-b-motif-memory','--updates','1500','--device','cuda','--memory','--init',ROOT/'run-a-event-baseline/best.pt','--lr','0.00015')
run('-m','supergroups.trancefusion_probe',ROOT/'run-b-motif-memory/best.pt','--out',ROOT/'run-b-motif-memory/validation-probe')


In [ ]:
run('-m','supergroups.trancefusion','--out',ROOT/'run-c-history-robustness','--updates','1500','--device','cuda','--memory','--init',ROOT/'run-b-motif-memory/best.pt','--lr','0.0001','--corruption','0.06')
run('-m','supergroups.trancefusion_probe',ROOT/'run-c-history-robustness/best.pt','--out',ROOT/'run-c-history-robustness/validation-probe')


## Final test
The original experiment selected C from validation loss and generated motif continuity. Keep these test songs untouched while tuning.


In [ ]:
run('-m','supergroups.trancefusion_probe',ROOT/'run-c-history-robustness/best.pt','--out',ROOT/'final-test','--split','test','--count','16')


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/trancefusion-guitarist-results','zip',ROOT)
files.download(archive)
